## Week 2 – Time-Windowed Feature Engineering

In this notebook, we aggregate raw ICU time-series data into patient-level
structured features using clinically motivated time windows to prepare for
patient similarity analysis.

### Time Window Selection

We begin with a 0–24 hour window from ICU admission, a commonly used
observation period in sepsis prediction literature, capturing early
physiologic trends while remaining interpretable.

In [21]:
import pandas as pd
import numpy as np

In [22]:
#load df
sepsis_df = pd.read_csv('/Volumes/ExtremeSSD/Projects/agentic-sepsis-patient-similarity/data/Dataset.csv')


In [23]:
sepsis_df['HospAdmTime'].describe()

count    1.552202e+06
mean    -5.612512e+01
std      1.622569e+02
min     -5.366860e+03
25%     -4.705000e+01
50%     -6.030000e+00
75%     -4.000000e-02
max      2.399000e+01
Name: HospAdmTime, dtype: float64

### Time Reference Semantics

HospAdmTime is defined as the number of hours from hospital admission to the
current ICU measurement. Negative values indicate hours after admission.
We therefore define early observation windows using absolute time since
admission (e.g., -24 to 0 hours).


In [24]:
df_24h = sepsis_df[(sepsis_df['HospAdmTime'] >= -24) & (sepsis_df['HospAdmTime'] <= 0)].copy()
df_24h.shape
df_24h['HospAdmTime'].describe()
df_24h.groupby('Patient_ID').size().describe()



count    27071.000000
mean        37.964242
std         22.336807
min          8.000000
25%         24.000000
50%         38.000000
75%         47.000000
max        336.000000
dtype: float64

In [25]:
df_24h.groupby('Patient_ID')['SepsisLabel'].max().value_counts(normalize=True)


SepsisLabel
0    0.929408
1    0.070592
Name: proportion, dtype: float64

In [26]:
sepsis_df.groupby('Patient_ID').size().describe()

count    40336.000000
mean        38.482001
std         22.795923
min          8.000000
25%         24.000000
50%         38.000000
75%         47.000000
max        336.000000
dtype: float64

### Feature Selection Rationale

We begin with a core set of vital signs and laboratory measurements that are
clinically relevant to sepsis, commonly available early in admission, and
frequently used in established scoring systems (e.g., SOFA). This approach
balances clinical interpretability with data completeness and provides a
robust baseline for patient similarity analysis.

In [27]:
numeric_features = [
    # Vitals
    'HR', 'O2Sat', 'Temp', 'SBP', 'DBP', 'MAP', 'Resp',
    
    # Labs
    'WBC', 'Lactate', 'Creatinine', 'Platelets'
]

missing_summary = (
    df_24h[numeric_features]
    .isna()
    .mean()
    .sort_values(ascending=False)
)

missing_summary


Lactate       0.976469
Platelets     0.941237
Creatinine    0.938425
WBC           0.936416
Temp          0.699261
DBP           0.348953
Resp          0.160868
SBP           0.157221
O2Sat         0.143027
MAP           0.137279
HR            0.108168
dtype: float64

### Feature Availability Considerations

While laboratory measurements are clinically relevant for sepsis, exploratory
analysis revealed substantial missingness (>90%) for most lab variables within
the first 24 hours after hospital admission. Consequently, we focus initial
patient representations on early vital signs, which are more consistently
available and better suited for similarity analysis in this time window.


In [28]:

#redefine numeric features for aggregation based on missingness
numeric_features = [
    'HR', 'MAP', 'O2Sat', 'SBP', 'DBP', 'Resp'
]


### Structured Feature Aggregation

We aggregate a clinically grounded subset of vital signs and laboratory
measurements within the first 24 hours after hospital admission. Features are
summarized per patient using mean, minimum, maximum, and standard deviation,
producing a fixed-length representation suitable for downstream similarity
analysis.


In [29]:
#aggregate per patient over 24h window
agg_df = (
    df_24h
    .groupby('Patient_ID')[numeric_features]
    .agg(['mean', 'min', 'max', 'std'])
)

agg_df.columns = [
    f"{feature}_{stat}_24h"
    for feature, stat in agg_df.columns
]

agg_df.reset_index(inplace=True)

agg_df.shape
agg_df.head()

,Patient_ID,HR_mean_24h,HR_min_24h,HR_max_24h,HR_std_24h,MAP_mean_24h,MAP_min_24h,MAP_max_24h,MAP_std_24h,O2Sat_mean_24h,...,SBP_max_24h,SBP_std_24h,DBP_mean_24h,DBP_min_24h,DBP_max_24h,DBP_std_24h,Resp_mean_24h,Resp_min_24h,Resp_max_24h,Resp_std_24h
0,1,101.571429,76.0,117.0,9.594378,87.261905,44.0,141.33,21.270465,91.477273,...,181.0,22.422482,NaN,NaN,NaN,NaN,24.820000,17.0,32.0,4.106689
1,4,102.444444,93.0,113.0,6.337212,67.147308,34.0,84.00,10.048494,98.203704,...,132.5,9.013857,51.428571,44.0,61.5,6.129554,18.884615,14.0,26.0,3.742480
2,5,73.916667,61.0,88.0,7.586697,87.088235,73.0,103.00,8.631764,97.500000,...,150.0,9.908252,NaN,NaN,NaN,NaN,16.500000,14.0,21.0,1.788854
3,6,100.000000,87.0,111.0,7.402702,88.110667,73.0,100.00,9.469159,98.437500,...,150.0,13.981961,NaN,NaN,NaN,NaN,26.583333,18.5,43.0,8.428622
4,7,120.363636,103.0,155.5,10.980090,74.125000,59.0,102.00,8.289896,95.409091,...,147.5,11.662463,59.627907,45.0,82.0,7.390545,20.909091,12.0,33.0,5.188620


In [30]:
agg_df.describe().T.head(10)


,count,mean,std,min,25%,50%,75%,max
Patient_ID,27071.0,57049.951165,50157.644918,1.000000,9564.500000,19155.000000,109334.500000,120000.000000
HR_mean_24h,27066.0,82.884256,14.840172,30.258065,72.241186,81.880694,92.454281,174.697674
HR_min_24h,27066.0,68.906691,13.938409,20.000000,59.500000,68.000000,78.000000,154.500000
HR_max_24h,27066.0,101.064446,19.336178,37.000000,88.000000,100.000000,113.000000,280.000000
HR_std_24h,27056.0,8.263614,4.069305,0.000000,5.571589,7.514567,10.096874,76.367532
MAP_mean_24h,26976.0,83.525932,13.028741,22.000000,74.082614,81.788996,91.324042,149.000000
MAP_min_24h,26976.0,65.342582,13.180771,20.000000,57.000000,64.000000,73.000000,140.000000
MAP_max_24h,26976.0,107.108661,23.375234,22.000000,92.500000,103.500000,117.000000,300.000000
MAP_std_24h,26856.0,9.999928,4.342390,0.000000,7.274778,9.290787,11.822188,133.392747
O2Sat_mean_24h,27058.0,97.149042,2.120592,27.000000,96.143201,97.390122,98.500000,100.000000


In [31]:
#Add SepsisLabel back to agg_df
labels = (
    df_24h[['Patient_ID', 'SepsisLabel']]
    .groupby('Patient_ID')
    .max()
    .reset_index()
)

patient_features = agg_df.merge(labels, on='Patient_ID', how='left')

patient_features.shape
patient_features.head()

#save processed feature dataframe
patient_features.to_csv(
    '/Volumes/ExtremeSSD/Projects/agentic-sepsis-patient-similarity/data/processed/patient_features_24h.csv',
    index=False
)

### Diagnostic Check 1: Missingness Audit

We evaluated missingness across engineered 24-hour features.
Missing values were primarily concentrated in standard deviation (`_std`) features,
which is expected when fewer than two measurements exist within a 24-hour window.

This confirms that missingness is structural (data sparsity), not a pipeline error.

In [32]:
#Add a binary indicator: 1 = missing at least once in 24h window, 0 = no missing values
missing_indicators = (
    df_24h
    .groupby('Patient_ID')[numeric_features]
    .apply(lambda x: x.isna().any().astype(int))
)

missing_indicators.columns = [
    f"{col}_missing_24h" for col in missing_indicators.columns
]

missing_indicators.reset_index(inplace=True)

#merge missing indicators with patient features aggregated dataframe
patient_features = agg_df.merge(
    missing_indicators,
    on='Patient_ID',
    how='left'
)
patient_features.shape
patient_features.head()
patient_features.isna().sum()



Patient_ID              0
HR_mean_24h             5
HR_min_24h              5
HR_max_24h              5
HR_std_24h             15
MAP_mean_24h           95
MAP_min_24h            95
MAP_max_24h            95
MAP_std_24h           215
O2Sat_mean_24h         13
O2Sat_min_24h          13
O2Sat_max_24h          13
O2Sat_std_24h          31
SBP_mean_24h          194
SBP_min_24h           194
SBP_max_24h           194
SBP_std_24h           252
DBP_mean_24h         5782
DBP_min_24h          5782
DBP_max_24h          5782
DBP_std_24h          5942
Resp_mean_24h          59
Resp_min_24h           59
Resp_max_24h           59
Resp_std_24h          109
HR_missing_24h          0
MAP_missing_24h         0
O2Sat_missing_24h       0
SBP_missing_24h         0
DBP_missing_24h         0
Resp_missing_24h        0
dtype: int64

In [34]:
patient_features.isna().sum().sum()
na_cols = patient_features.columns[patient_features.isna().any()]
len(na_cols), na_cols.tolist()[:10]

(24,
 ['HR_mean_24h',
  'HR_min_24h',
  'HR_max_24h',
  'HR_std_24h',
  'MAP_mean_24h',
  'MAP_min_24h',
  'MAP_max_24h',
  'MAP_std_24h',
  'O2Sat_mean_24h',
  'O2Sat_min_24h'])

### Handling Missing Values After Aggregation
Aggregating time-series data over the first 24 hours can introduce missing
values, particularly when a patient has limited measurements for a given
vital sign. For example, standard deviation is undefined when only a single
measurement is available. These cases are handled explicitly to preserve
clinical signal while enabling distance-based similarity modeling.

#### Standard Deviation Features

Missing values in standard deviation features indicate a lack of observed
variability during the 24-hour window. These are set to zero, representing
no observed variation rather than missing physiological information.


In [35]:
# Handle NaNs introduced by aggregation (e.g., std with single observation)
std_cols = [c for c in patient_features.columns if c.endswith('_std_24h')]

len(std_cols), std_cols[:5]
#fill std NaNs with 0
patient_features[std_cols] = patient_features[std_cols].fillna(0)
patient_features.isna().sum().sum()

np.int64(18444)

#### Remaining Missing Aggregate
For patients without any recorded measurements for a given vital sign within
the first 24 hours, remaining missing aggregate values are imputed using the
feature-wise mean. Missingness indicators are retained to ensure the model
can distinguish between imputed and observed values.


In [37]:
#Fill remaining NaNs with column mean - min, max, mean values
patient_features = patient_features.fillna(patient_features.mean())
#verify no remaining NaNs
patient_features.isna().sum().sum()

np.int64(0)

In [40]:
patient_features.to_csv(
    "/Volumes/ExtremeSSD/Projects/agentic-sepsis-patient-similarity/data/processed/patient_features_24h_vitals.csv",
    index=False
)

In [42]:
#remove missing columns before normalizing dataset
missing_cols = [c for c in patient_features.columns if c.endswith('_missing_24h')]
df = patient_features.drop(columns=missing_cols)

df.to_csv(
    "/Volumes/ExtremeSSD/Projects/agentic-sepsis-patient-similarity/data/processed/patient_features_24h_vitals_clean.csv",
    index=False
)

In [43]:
patient_features_clean=pd.read_csv('/Volumes/ExtremeSSD/Projects/agentic-sepsis-patient-similarity/data/processed/patient_features_24h_vitals_clean.csv')

In [44]:
#Normalize features using Scaling
from sklearn.preprocessing import StandardScaler

scale_cols = [ c for c in patient_features_clean.columns
    if c not in ['Patient_ID', 'SepsisLabel']]


scaler = StandardScaler()
patient_features_clean[scale_cols] = scaler.fit_transform(
    patient_features_clean[scale_cols]
)

patient_features_clean.head()

,Patient_ID,HR_mean_24h,HR_min_24h,HR_max_24h,HR_std_24h,MAP_mean_24h,MAP_min_24h,MAP_max_24h,MAP_std_24h,O2Sat_mean_24h,...,SBP_max_24h,SBP_std_24h,DBP_mean_24h,DBP_min_24h,DBP_max_24h,DBP_std_24h,Resp_mean_24h,Resp_min_24h,Resp_max_24h,Resp_std_24h
0,1,1.259368,0.508960,0.824223,0.327873,0.287258,-1.622100,1.466603,2.570669,-2.675308,...,1.116376,1.640263,-1.431256e-15,7.447770e-16,0.000000,-1.369127,1.911626,1.238906,0.942669,0.533060
1,4,1.318203,1.728747,0.617334,-0.471874,-1.259350,-2.382130,-0.990353,0.028988,0.497472,...,-0.794244,-0.826913,-1.416962e+00,-7.712640e-01,-1.351663,-0.081593,0.123405,0.370217,0.012606,0.303229
2,5,-0.604345,-0.567322,-0.675723,-0.165082,0.273905,0.581987,-0.176082,-0.291889,0.165543,...,-0.104845,-0.662345,-1.431256e-15,7.447770e-16,0.000000,-1.369127,-0.595035,0.370217,-0.762446,-0.929587
3,6,1.153467,1.298234,0.513889,-0.210259,0.352520,0.581987,-0.304651,-0.102227,0.607751,...,-0.104845,0.087214,-1.431256e-15,7.447770e-16,0.000000,-1.369127,2.442886,1.673250,2.647783,3.260374
4,7,2.525815,2.446268,2.815530,0.668113,-0.722836,-0.482055,-0.218939,-0.369320,-0.820715,...,-0.203331,-0.339572,-5.911604e-01,-6.664460e-01,-0.209627,0.183283,0.733342,-0.208909,1.097679,1.215802


In [45]:
#merge with sepsis labels
# Use full temporal data to determine patient-level sepsis outcome
labels = (
    sepsis_df[['Patient_ID', 'SepsisLabel']]
    .groupby('Patient_ID')
    .max()
    .reset_index()
)

patient_features_clean = patient_features_clean.merge(
    labels, on='Patient_ID', how='left'
)


In [46]:
patient_features_clean.to_csv(
    "/Volumes/ExtremeSSD/Projects/agentic-sepsis-patient-similarity/data/processed/normalized_patient_features_24h_vitals_clean.csv",
    index=False
)